# 🚀 SC2-Hailo8-Transformer: Entrenamiento Multi-Raza Balanceado y Compilación `.hef`

Este notebook entrena un modelo **Action-Chunking Transformer 4D Estático** para jugar **StarCraft II** (Terran, Protoss y Zerg) diseñado específicamente para ejecutarse en una **Raspberry Pi 5 con HAT Hailo-8 (26 TOPS INT8)**.

### Características clave frente a AlphaStar:
1. **100% Tensores 4D Estáticos (`[1, 12, 16, 16]` -> `[1, 66, 1, 4]`)**: Sin *Pointer Networks* dinámicos ni LSTMs recurrentes que fallen al compilar a `.hef`.
2. **Corrección de Desbalanceo Severo**: Descarga **`yxzhang2024/SC2-Dynamics-50K`** desde Hugging Face, analiza la distribución **(Raza × Acción)** y aplica **`WeightedRandomSampler`** + **`ClassBalancedFocalLoss`** ($\beta=0.999, \gamma=2.0$).
3. **Exportación ONNX Opset 16 Autocontenida**: Genera un único archivo `.onnx` sin fragmentación `.onnx.data` ni `LayerNorm` 1D problemáticos.
4. **Compilación Directa a `.hef`**: Usa tu instalador `hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl` en Google Drive con sintaxis `.alls` válida y calibración INT8 real.

In [ ]:
# 1. Montar Google Drive e instalar dependencias para entrenamiento y exportación ONNX
from google.colab import drive
drive.mount('/content/drive')

!pip install -q huggingface_hub onnx onnxruntime onnxscript matplotlib

In [ ]:
import os
import re
import json
import math
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Carpeta en Google Drive donde guardaremos el ONNX, datos de calibración y el binario .hef
DRIVE_HAILO_DIR = "/content/drive/MyDrive/hailo_files"
os.makedirs(DRIVE_HAILO_DIR, exist_ok=True)

# Dimensiones estáticas NCHW para Hailo-8
GRID_H = 16
GRID_W = 16
IN_CHANNELS = 12       # 6 canales espaciales (minimapa) + 6 canales de economía/raza/tiempo
ACTION_CHUNK_SIZE = 4  # Horizonte K=4 acciones futuras por cada inferencia

RACES = {"Terran": 0, "Protoss": 1, "Zerg": 2, "Random": 3}
ID_TO_RACE = {0: "Terran", 1: "Protoss", 2: "Zerg"}

ACTION_VOCAB = [
    "NO_OP",                        # 0
    "HARVEST_GATHER_MINERALS",      # 1
    "HARVEST_GATHER_GAS",           # 2
    "TRAIN_WORKER",                 # 3
    "BUILD_SUPPLY",                 # 4
    "BUILD_GAS_EXTRACTOR",          # 5
    "BUILD_EXPAND_BASE",            # 6
    "BUILD_BARRACKS_GATEWAY_POOL",  # 7
    "BUILD_FACTORY_ROBO_ROACH",     # 8
    "BUILD_STARPORT_STARGATE_SPIRE",# 9
    "BUILD_TECH_LAB_CORE_LAIR",     # 10
    "BUILD_DEFENSE_STRUCTURE",      # 11
    "TRAIN_BASIC_INFANTRY",         # 12
    "TRAIN_RANGED_CORE",            # 13
    "TRAIN_SPECIALIST_UNIT",        # 14
    "TRAIN_SIEGE_HEAVY",            # 15
    "TRAIN_AIR_UNIT",               # 16
    "RESEARCH_COMBAT_UPGRADE",      # 17
    "RESEARCH_WEAPONS_ARMOR",       # 18
    "MACRO_CHRONO_MULE_INJECT",     # 19
    "MOVE_ARMY_POSITION",           # 20
    "MOVE_WORKER_SCOUT",            # 21
    "ATTACK_MOVE_COORDINATE",       # 22
    "FOCUS_FIRE_TARGET",            # 23
    "MICRO_KITE_RETREAT",           # 24
    "MICRO_USE_STIM_ABILITY",       # 25
    "MICRO_SIEGE_TRANSFORM",        # 26
    "MICRO_HEAL_REPAIR_SHIELD",     # 27
    "DEFEND_MAIN_NATURAL",          # 28
    "HARASS_ENEMY_WORKERS",         # 29
    "RALLY_PRODUCTION",             # 30
    "CANCEL_OR_HOLD_POSITION",      # 31
]
NUM_ACTIONS = len(ACTION_VOCAB)
ACTION_TO_ID = {name: i for i, name in enumerate(ACTION_VOCAB)}
ID_TO_ACTION = {i: name for i, name in enumerate(ACTION_VOCAB)}
STEP_OUTPUT_DIM = NUM_ACTIONS + GRID_W + GRID_H + 2  # 32 + 16 + 16 + 2 = 66 logits por paso

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo activo: {device} | Salida por paso: {STEP_OUTPUT_DIM} logits x {ACTION_CHUNK_SIZE} chunks")

In [ ]:
# 2. Descarga y Procesamiento de 'yxzhang2024/SC2-Dynamics-50K' desde Hugging Face
from huggingface_hub import hf_hub_download

def map_raw_sc2_command_to_action_id(subject: str, command: str, target: str) -> int:
    s, c, t = subject.lower(), command.lower(), target.lower()
    if "harvest" in c or ("smartcmd" in c and any(k in t for k in ["mineral", "vespene", "geyser"])):
        if any(k in t for k in ["gas", "vespene", "refinery", "assimilator", "extractor"]):
            return ACTION_TO_ID["HARVEST_GATHER_GAS"]
        return ACTION_TO_ID["HARVEST_GATHER_MINERALS"]
    if any(k in c for k in ["train_scv", "train_probe", "train_drone", "morph_drone"]):
        return ACTION_TO_ID["TRAIN_WORKER"]
    if any(k in c for k in ["supplydepot", "pylon", "overlord"]):
        return ACTION_TO_ID["BUILD_SUPPLY"]
    if any(k in c for k in ["refinery", "assimilator", "extractor"]):
        return ACTION_TO_ID["BUILD_GAS_EXTRACTOR"]
    if "build" in c or "morph" in c:
        if any(k in c for k in ["commandcenter", "nexus", "hatchery", "orbital", "planetary"]):
            return ACTION_TO_ID["BUILD_EXPAND_BASE"]
        if any(k in c for k in ["barracks", "gateway", "spawningpool"]):
            return ACTION_TO_ID["BUILD_BARRACKS_GATEWAY_POOL"]
        if any(k in c for k in ["factory", "robotics", "roachwarren", "banelingnest", "hydraliskden"]):
            return ACTION_TO_ID["BUILD_FACTORY_ROBO_ROACH"]
        if any(k in c for k in ["starport", "stargate", "spire"]):
            return ACTION_TO_ID["BUILD_STARPORT_STARGATE_SPIRE"]
        if any(k in c for k in ["techlab", "reactor", "cyberneticscore", "twilight", "lair", "hive", "infestation"]):
            return ACTION_TO_ID["BUILD_TECH_LAB_CORE_LAIR"]
        if any(k in c for k in ["bunker", "turret", "cannon", "battery", "spine", "spore"]):
            return ACTION_TO_ID["BUILD_DEFENSE_STRUCTURE"]
    if "train" in c or "warp" in c or "morph" in c:
        if any(k in c for k in ["marine", "zealot", "zergling"]):
            return ACTION_TO_ID["TRAIN_BASIC_INFANTRY"]
        if any(k in c for k in ["marauder", "stalker", "adept", "roach", "hydralisk", "ravager"]):
            return ACTION_TO_ID["TRAIN_RANGED_CORE"]
        if any(k in c for k in ["reaper", "ghost", "sentry", "templar", "baneling", "infestor", "viper", "queen"]):
            return ACTION_TO_ID["TRAIN_SPECIALIST_UNIT"]
        if any(k in c for k in ["tank", "thor", "hellion", "cyclone", "immortal", "colossus", "disruptor", "ultralisk", "lurker"]):
            return ACTION_TO_ID["TRAIN_SIEGE_HEAVY"]
        if any(k in c for k in ["medivac", "viking", "liberator", "banshee", "battlecruiser", "phoenix", "voidray", "oracle", "carrier", "tempest", "mutalisk", "corruptor", "broodlord"]):
            return ACTION_TO_ID["TRAIN_AIR_UNIT"]
    if "research" in c or "upgrade" in c:
        if any(k in c for k in ["weapon", "armor", "shield", "melee", "missile", "carapace", "flyer"]):
            return ACTION_TO_ID["RESEARCH_WEAPONS_ARMOR"]
        return ACTION_TO_ID["RESEARCH_COMBAT_UPGRADE"]
    if any(k in c for k in ["mule", "chrono", "inject", "spawnlarva", "transfusion", "scanner"]):
        return ACTION_TO_ID["MACRO_CHRONO_MULE_INJECT"]
    if any(k in c for k in ["stim", "blink", "forcefield", "guardian", "storm", "corrosive", "emp", "snipe", "fungal"]):
        return ACTION_TO_ID["MICRO_USE_STIM_ABILITY"]
    if any(k in c for k in ["siege", "unsiege", "burrow", "unburrow", "assault", "fighter", "defender"]):
        return ACTION_TO_ID["MICRO_SIEGE_TRANSFORM"]
    if any(k in c for k in ["repair", "heal", "restore"]):
        return ACTION_TO_ID["MICRO_HEAL_REPAIR_SHIELD"]
    if "rally" in c:
        return ACTION_TO_ID["RALLY_PRODUCTION"]
    if "cancel" in c or "hold" in c or "stop" in c:
        return ACTION_TO_ID["CANCEL_OR_HOLD_POSITION"]
    if "attack" in c:
        if "[" in t and not t.startswith("("):
            if any(w in t for w in ["scv", "probe", "drone"]):
                return ACTION_TO_ID["HARASS_ENEMY_WORKERS"]
            return ACTION_TO_ID["FOCUS_FIRE_TARGET"]
        return ACTION_TO_ID["ATTACK_MOVE_COORDINATE"]
    if "move" in c or "smartcmd" in c:
        if any(w in s for w in ["scv", "probe", "drone"]):
            return ACTION_TO_ID["HARVEST_GATHER_MINERALS"]
        if "overlord" in s:
            return ACTION_TO_ID["MOVE_WORKER_SCOUT"]
        return ACTION_TO_ID["MOVE_ARMY_POSITION"]
    return ACTION_TO_ID["NO_OP"]


def parse_sc2_dynamics_sample(sample_text: str, map_size: float = 64.0):
    if "#### Current Observation:" not in sample_text:
        return None
    obs_part, *act_parts = sample_text.split("#### Action Sequence")
    act_part = act_parts[0] if act_parts else ""
    tensor = np.zeros((IN_CHANNELS, GRID_H, GRID_W), dtype=np.float32)

    time_match = re.search(r"Time:\s*(\d+):(\d+)", obs_part)
    game_sec = (int(time_match.group(1)) * 60 + int(time_match.group(2))) if time_match else 0
    race_match = re.search(r"Race:\s*(Terran|Protoss|Zerg)", obs_part)
    enemy_race_match = re.search(r"Enemy Race:\s*(Terran|Protoss|Zerg)", obs_part)
    self_race = RACES.get(race_match.group(1), 0) if race_match else 0
    enemy_race = RACES.get(enemy_race_match.group(1), 0) if enemy_race_match else 0

    min_match = re.search(r"Minerals:\s*(\d+)\s*\(\+(\d+)/min\)", obs_part)
    minerals = float(min_match.group(1)) if min_match else 50.0
    min_rate = float(min_match.group(2)) if min_match else 0.0

    gas_match = re.search(r"Gas:\s*(\d+)\s*\(\+(\d+)/min\)", obs_part)
    gas = float(gas_match.group(1)) if gas_match else 0.0

    sup_match = re.search(r"Supply:\s*(\d+)/(\d+)\s*\(Army:\s*(\d+),\s*Workers:\s*(\d+)\)", obs_part)
    sup_used = float(sup_match.group(1)) if sup_match else 12.0
    army_sup = float(sup_match.group(3)) if sup_match else 0.0
    worker_sup = float(sup_match.group(4)) if sup_match else 12.0

    tensor[6, :, :] = np.clip(minerals / 2000.0, 0.0, 1.0)
    tensor[7, :, :] = np.clip(gas / 1500.0, 0.0, 1.0)
    tensor[8, :, :] = np.clip(sup_used / 200.0, 0.0, 1.0)
    tensor[9, :, :] = np.clip(army_sup / 120.0, 0.0, 1.0)
    tensor[10, :, :] = np.clip(worker_sup / 90.0, 0.0, 1.0)
    tensor[11, 0:4, :] = np.clip(game_sec / 1200.0, 0.0, 1.0)
    tensor[11, 4:8, :] = np.clip(min_rate / 2500.0, 0.0, 1.0)
    tensor[11, 8:12, :] = (self_race + 1) / 3.0
    tensor[11, 12:16, :] = (enemy_race + 1) / 3.0

    sections = re.split(r"\[(My Units|My Structures|Visible Hostiles|Queue)\]", obs_part)
    current_sec = ""
    coord_pattern = re.compile(r"at\s*\((\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\)(?:\s*\(HP:(\d+)%)?")
    for part in sections:
        if part in ("My Units", "My Structures", "Visible Hostiles", "Queue"):
            current_sec = part
            continue
        if current_sec == "Queue":
            active_tasks = len([l for l in part.splitlines() if l.strip().startswith("-") and "Empty" not in l])
            tensor[5, :, :] = np.clip(active_tasks / 10.0, 0.0, 1.0)
            continue
        for m in coord_pattern.finditer(part):
            x_raw, y_raw = float(m.group(1)), float(m.group(2))
            hp_pct = float(m.group(3)) / 100.0 if m.group(3) else 1.0
            gx = int(np.clip((x_raw / map_size) * GRID_W, 0, GRID_W - 1))
            gy = int(np.clip((y_raw / map_size) * GRID_H, 0, GRID_H - 1))
            if current_sec == "My Units":
                tensor[0, gy, gx] = min(1.0, tensor[0, gy, gx] + 0.25)
                tensor[1, gy, gx] = hp_pct
            elif current_sec == "My Structures":
                tensor[2, gy, gx] = min(1.0, tensor[2, gy, gx] + 0.5)
            elif current_sec == "Visible Hostiles":
                tensor[3, gy, gx] = min(1.0, tensor[3, gy, gx] + 0.25)
                tensor[4, gy, gx] = hp_pct

    action_lines = [l.strip() for l in act_part.splitlines() if l.strip().startswith("- +")]
    parsed_actions = []
    act_line_re = re.compile(r"-\s*\+[\d.]+s:\s*([^\-]+?)\s*-\s*([^\-]+?)(?:\s*-\s*(.+))?$")
    target_coord_re = re.compile(r"\((\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\)")

    for line in action_lines:
        m = act_line_re.match(line)
        if not m:
            continue
        subj, cmd, targ = m.group(1).strip(), m.group(2).strip(), (m.group(3) or "").strip()
        act_id = map_raw_sc2_command_to_action_id(subj, cmd, targ)
        tx, ty = GRID_W // 2, GRID_H // 2
        cm = target_coord_re.search(targ)
        if cm:
            tx = int(np.clip((float(cm.group(1)) / map_size) * GRID_W, 0, GRID_W - 1))
            ty = int(np.clip((float(cm.group(2)) / map_size) * GRID_H, 0, GRID_H - 1))
        queue_flag = 1 if ("smartcmd" in cmd.lower() and len(parsed_actions) > 0) else 0
        parsed_actions.append((act_id, tx, ty, queue_flag))

    # Agrupar por tipo de acción (act_id) para que múltiples unidades ejecutando Attack al mismo tiempo
    # no llenen los 4 slots del Action-Chunk y descarten las órdenes de construir estructuras y entrenar unidades
    unique_by_act_id = {}
    for item in parsed_actions:
        act_id, tx, ty, q_flag = item
        if act_id not in unique_by_act_id:
            unique_by_act_id[act_id] = item
        elif (unique_by_act_id[act_id][1], unique_by_act_id[act_id][2]) == (GRID_W // 2, GRID_H // 2) and (tx, ty) != (GRID_W // 2, GRID_H // 2):
            unique_by_act_id[act_id] = item

    dedup_actions = list(unique_by_act_id.values())
    macro_build = [a for a in dedup_actions if a[0] in (4, 5, 6, 7, 8, 9, 10, 11, 17, 18, 19)]
    unit_prod = [a for a in dedup_actions if a[0] in (3, 12, 13, 14, 15, 16)]
    combat_micro = [a for a in dedup_actions if a[0] in (22, 23, 24, 25, 26, 27, 28, 29, 30, 31)]
    econ_move = [a for a in dedup_actions if a[0] in (1, 2, 20, 21)]
    ordered = (macro_build + unit_prod + combat_micro + econ_move)[:ACTION_CHUNK_SIZE]

    fill_act = ACTION_TO_ID["HARVEST_GATHER_MINERALS"] if game_sec < 180 else ACTION_TO_ID["NO_OP"]
    while len(ordered) < ACTION_CHUNK_SIZE:
        ordered.append((fill_act, GRID_W // 2, GRID_H // 2, 0))

    return tensor, np.array(ordered[:ACTION_CHUNK_SIZE], dtype=np.int64), self_race


class SC2ActionChunkDataset(Dataset):
    def __init__(self, states: np.ndarray, actions: np.ndarray, races: np.ndarray):
        self.states = states.astype(np.float32)
        self.actions = actions.astype(np.int64)
        self.races = races.astype(np.int64)

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.states[idx]),
            torch.from_numpy(self.actions[idx]),
            torch.tensor(self.races[idx], dtype=torch.long),
        )

    def build_balancer(self):
        primary_actions = self.actions[:, 0, 0]
        all_actions = self.actions[:, :, 0].reshape(-1)
        race_counts = Counter(self.races.tolist())
        act_counts = Counter(all_actions.tolist())
        pair_counts = Counter(zip(self.races.tolist(), primary_actions.tolist()))

        sample_weights = np.zeros(len(self.states), dtype=np.float64)
        for i in range(len(self.states)):
            key = (int(self.races[i]), int(primary_actions[i]))
            sample_weights[i] = 1.0 / np.sqrt(max(1, pair_counts[key]))
        sample_weights /= sample_weights.mean()

        sampler = WeightedRandomSampler(
            weights=torch.from_numpy(sample_weights).double(),
            num_samples=len(sample_weights),
            replacement=True,
        )

        beta = 0.999
        class_weights = np.ones(NUM_ACTIONS, dtype=np.float32)
        for c in range(NUM_ACTIONS):
            n_c = max(1, act_counts.get(c, 0))
            class_weights[c] = (1.0 - beta) / (1.0 - np.power(beta, n_c))
        class_weights /= np.mean(class_weights)

        return sampler, torch.from_numpy(class_weights), race_counts, act_counts

In [ ]:
# 3. Descargar Dataset Real de StarCraft II (V2 Limpio + Multi-Matchup) y Analizar el Desbalanceo
MAX_SAMPLES = 30000
CACHE_NPZ = os.path.join(DRIVE_HAILO_DIR, f"sc2_dynamics_clean_v2_{MAX_SAMPLES}.npz")

def generate_multi_matchup_augmentation(num_samples: int = 6000, seed: int = 42):
    """Genera muestras multi-matchup (TvZ, TvP, TvT, PvZ, ZvT) con las 32 acciones completas."""
    rng = np.random.default_rng(seed)
    states = np.zeros((num_samples, IN_CHANNELS, GRID_H, GRID_W), dtype=np.float32)
    actions = np.zeros((num_samples, ACTION_CHUNK_SIZE, 4), dtype=np.int64)
    races = np.zeros(num_samples, dtype=np.int64)
    for i in range(num_samples):
        race = 0 if i < int(num_samples * 0.6) else (i % 3)
        enemy_race = (i + 1) % 3
        races[i] = race
        phase = rng.choice([0, 1, 2], p=[0.35, 0.40, 0.25])
        game_progress = rng.uniform(0.02, 0.25) if phase == 0 else (rng.uniform(0.25, 0.65) if phase == 1 else rng.uniform(0.65, 1.0))
        minerals = rng.uniform(50, 350) if phase == 0 else rng.uniform(150, 1400)
        gas = rng.uniform(0, 120) if phase == 0 else rng.uniform(80, 900)
        workers = rng.uniform(12, 26) if phase == 0 else rng.uniform(32, 72)
        army = rng.uniform(0, 12) if phase == 0 else rng.uniform(20, 100)
        supply = min(200.0, workers + army)
        ally_bx, ally_by = (3, 3) if rng.random() < 0.5 else (12, 12)
        enemy_bx, enemy_by = (15 - ally_bx, 15 - ally_by)
        states[i, 0, max(0, ally_by - 2):min(16, ally_by + 3), max(0, ally_bx - 2):min(16, ally_bx + 3)] = rng.uniform(0.3, 1.0)
        states[i, 1, max(0, ally_by - 2):min(16, ally_by + 3), max(0, ally_bx - 2):min(16, ally_bx + 3)] = rng.uniform(0.6, 1.0)
        states[i, 2, ally_by, ally_bx] = 1.0
        under_attack = rng.random() < 0.28
        ex = ally_bx + rng.integers(-2, 3) if under_attack else enemy_bx + rng.integers(-2, 3)
        ey = ally_by + rng.integers(-2, 3) if under_attack else enemy_by + rng.integers(-2, 3)
        ex, ey = int(np.clip(ex, 0, 15)), int(np.clip(ey, 0, 15))
        states[i, 3, ey, ex] = rng.uniform(0.4, 1.0)
        states[i, 4, ey, ex] = rng.uniform(0.4, 1.0)
        states[i, 5, :, :] = rng.uniform(0.0, 0.7)
        states[i, 6, :, :] = minerals / 2000.0
        states[i, 7, :, :] = gas / 1500.0
        states[i, 8, :, :] = supply / 200.0
        states[i, 9, :, :] = army / 120.0
        states[i, 10, :, :] = workers / 90.0
        states[i, 11, 0:4, :] = game_progress
        states[i, 11, 4:8, :] = min(1.0, workers / 66.0)
        states[i, 11, 8:12, :] = (race + 1) / 3.0
        states[i, 11, 12:16, :] = (enemy_race + 1) / 3.0
        if under_attack:
            chunk_acts = [25, 26, 23, 28]
        elif phase == 0:
            chunk_acts = list(rng.choice([3, 4, 7, 5, 12, 1, 2, 19], size=4, replace=False))
        elif phase == 1:
            chunk_acts = list(rng.choice([6, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 22], size=4, replace=False))
        else:
            chunk_acts = list(rng.choice([15, 16, 18, 25, 26, 22, 23, 24, 27, 29], size=4, replace=False))
        for k, act_id in enumerate(chunk_acts[:ACTION_CHUNK_SIZE]):
            tx, ty = (ex, ey) if act_id in (22, 23, 25, 26, 29) else (ally_bx, ally_by)
            actions[i, k] = [int(act_id), tx, ty, 0]
    return states, actions, races

if os.path.exists(CACHE_NPZ):
    print(f"📂 Cargando dataset preprocesado desde Google Drive: {CACHE_NPZ}")
    loaded = np.load(CACHE_NPZ)
    states_all, actions_all, races_all = loaded["states"], loaded["actions"], loaded["races"]
else:
    print("🌐 Descargando 'yxzhang2024/SC2-Dynamics-50K' desde Hugging Face...")
    states_list, actions_list, races_list = [], [], []
    for split_file in ["wm_train_horizon5.json", "wm_val_horizon5.json"]:
        path = hf_hub_download(
            repo_id="yxzhang2024/SC2-Dynamics-50K",
            filename=split_file,
            repo_type="dataset",
        )
        with open(path, "r", encoding="utf-8") as f:
            records = json.load(f)
        for rec in records:
            msgs = rec.get("messages", [])
            if not msgs:
                continue
            parsed = parse_sc2_dynamics_sample(msgs[0].get("content", ""))
            if parsed is not None:
                st, ac, rc = parsed
                states_list.append(st)
                actions_list.append(ac)
                races_list.append(rc)
                if len(states_list) >= MAX_SAMPLES:
                    break
        if len(states_list) >= MAX_SAMPLES:
            break

    states_real = np.stack(states_list, axis=0)
    actions_real = np.stack(actions_list, axis=0)
    races_real = np.array(races_list, dtype=np.int64)
    st_aug, ac_aug, rc_aug = generate_multi_matchup_augmentation(num_samples=6000)
    states_all = np.concatenate([states_real, st_aug], axis=0)
    actions_all = np.concatenate([actions_real, ac_aug], axis=0)
    races_all = np.concatenate([races_real, rc_aug], axis=0)
    np.savez_compressed(CACHE_NPZ, states=states_all, actions=actions_all, races=races_all)
    print(f"💾 Guardado caché V2 limpio + multi-matchup en Drive: {CACHE_NPZ} ({len(states_all)} muestras)")

perm = np.random.default_rng(42).permutation(len(states_all))
split = int(len(states_all) * 0.85)
train_ds = SC2ActionChunkDataset(states_all[perm[:split]], actions_all[perm[:split]], races_all[perm[:split]])
val_ds = SC2ActionChunkDataset(states_all[perm[split:]], actions_all[perm[split:]], races_all[perm[split:]])

train_sampler, class_weights, race_counts, act_counts = train_ds.build_balancer()

print(f"\n📊 Total Muestras: {len(states_all)} (Train: {len(train_ds)}, Val: {len(val_ds)})")
print("📊 Distribución por Raza en Train:", {ID_TO_RACE.get(k, k): v for k, v in race_counts.items()})
print("\n🔝 Top 8 Acciones más frecuentes (Antes del balanceo):")
for act_id, cnt in act_counts.most_common(8):
    print(f"  - [{act_id:02d}] {ID_TO_ACTION[act_id]:<30}: {cnt:6d} ocurrencias | Peso Focal: {class_weights[act_id]:.3f}x")

print("\n⚡ Acciones Estratégicas/Combate con refuerzo de peso (Class-Balanced Focal Loss):")
for act_id, cnt in act_counts.most_common()[-8:]:
    print(f"  - [{act_id:02d}] {ID_TO_ACTION[act_id]:<30}: {cnt:6d} ocurrencias | Peso Focal: {class_weights[act_id]:.3f}x")

In [ ]:
# 4. Arquitectura SC2HailoTransformer (100% NCHW 4D Estática para Hailo-8)
class HailoAttentionBlock2D(nn.Module):
    def __init__(self, d_model: int = 128, n_heads: int = 4, mlp_ratio: int = 2, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.norm1 = nn.BatchNorm2d(d_model)
        self.q_proj = nn.Conv2d(d_model, d_model, kernel_size=1, bias=True)
        self.k_proj = nn.Conv2d(d_model, d_model, kernel_size=1, bias=True)
        self.v_proj = nn.Conv2d(d_model, d_model, kernel_size=1, bias=True)
        self.out_proj = nn.Conv2d(d_model, d_model, kernel_size=1, bias=True)
        self.attn_drop = nn.Dropout(dropout)

        hidden_dim = d_model * mlp_ratio
        self.norm2 = nn.BatchNorm2d(d_model)
        self.ffn = nn.Sequential(
            nn.Conv2d(d_model, hidden_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv2d(hidden_dim, d_model, kernel_size=1, bias=True),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        N = H * W
        h = self.norm1(x)
        q = self.q_proj(h).reshape(B, self.n_heads, self.head_dim, N).permute(0, 1, 3, 2)
        k = self.k_proj(h).reshape(B, self.n_heads, self.head_dim, N)
        v = self.v_proj(h).reshape(B, self.n_heads, self.head_dim, N).permute(0, 1, 3, 2)

        attn = F.softmax(torch.matmul(q, k) * self.scale, dim=-1)
        attn = self.attn_drop(attn)

        out = torch.matmul(attn, v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        x = x + self.out_proj(out)
        x = x + self.ffn(self.norm2(x))
        return x


class SC2HailoTransformer(nn.Module):
    def __init__(self, in_channels: int = IN_CHANNELS, d_model: int = 128, n_heads: int = 4, num_layers: int = 4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, d_model, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(d_model),
            nn.ReLU(inplace=True),
        )
        self.pos_embed = nn.Parameter(torch.randn(1, d_model, GRID_H, GRID_W) * 0.02)
        self.blocks = nn.ModuleList([HailoAttentionBlock2D(d_model=d_model, n_heads=n_heads) for _ in range(num_layers)])
        self.final_norm = nn.BatchNorm2d(d_model)
        self.chunk_pool = nn.Sequential(
            nn.Conv2d(d_model, d_model, kernel_size=(GRID_H, ACTION_CHUNK_SIZE), stride=(GRID_H, ACTION_CHUNK_SIZE), bias=False),
            nn.BatchNorm2d(d_model),
            nn.ReLU(inplace=True),
        )
        self.action_head = nn.Conv2d(d_model, STEP_OUTPUT_DIM, kernel_size=1, bias=True)

    def forward(self, obs_tensor: torch.Tensor) -> torch.Tensor:
        x = self.stem(obs_tensor) + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        x = self.final_norm(x)
        chunks_feat = self.chunk_pool(x)
        return self.action_head(chunks_feat)  # [B, 66, 1, 4]

    @staticmethod
    def split_predictions(out_tensor: torch.Tensor):
        seq = out_tensor.squeeze(2).permute(0, 2, 1)  # [B, 4, 66]
        c1, c2, c3 = NUM_ACTIONS, NUM_ACTIONS + GRID_W, NUM_ACTIONS + GRID_W + GRID_H
        return seq[:, :, :c1], seq[:, :, c1:c2], seq[:, :, c2:c3], seq[:, :, c3:]


class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_weights: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.gamma = gamma

    def forward(self, raw_out: torch.Tensor, target_chunks: torch.Tensor):
        act_logits, tx_logits, ty_logits, q_logits = SC2HailoTransformer.split_predictions(raw_out)
        t_act = target_chunks[:, :, 0].reshape(-1)
        t_tx = target_chunks[:, :, 1].reshape(-1)
        t_ty = target_chunks[:, :, 2].reshape(-1)
        t_q = target_chunks[:, :, 3].reshape(-1)

        flat_act = act_logits.reshape(-1, NUM_ACTIONS)
        ce_act = F.cross_entropy(flat_act, t_act, reduction="none")
        pt = torch.exp(-ce_act)
        focal_loss = (self.class_weights[t_act] * ((1.0 - pt) ** self.gamma) * ce_act).mean()

        loss_tx = F.cross_entropy(tx_logits.reshape(-1, GRID_W), t_tx)
        loss_ty = F.cross_entropy(ty_logits.reshape(-1, GRID_H), t_ty)
        loss_q = F.cross_entropy(q_logits.reshape(-1, 2), t_q)
        total = focal_loss + 0.5 * (loss_tx + loss_ty) + 0.25 * loss_q

        with torch.no_grad():
            acc_act = (flat_act.argmax(dim=-1) == t_act).float().mean().item()
            acc_xy = 0.5 * (
                (tx_logits.reshape(-1, GRID_W).argmax(dim=-1) == t_tx).float().mean().item() +
                (ty_logits.reshape(-1, GRID_H).argmax(dim=-1) == t_ty).float().mean().item()
            )
        return total, acc_act, acc_xy

In [ ]:
# 5. Entrenamiento en GPU con WeightedRandomSampler + Class-Balanced Focal Loss
BATCH_SIZE = 64
EPOCHS = 15
LR = 3e-4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, drop_last=True, num_workers=2, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(device.type == 'cuda'))

model = SC2HailoTransformer(in_channels=IN_CHANNELS, d_model=128, n_heads=4, num_layers=4).to(device)
criterion = ClassBalancedFocalLoss(class_weights=class_weights.to(device), gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float("inf")
weights_path = os.path.join(DRIVE_HAILO_DIR, "sc2_hailo8_transformer_best.pth")

print(f"🔥 Iniciando entrenamiento en {device} ({EPOCHS} épocas)...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss, tr_acc_act, tr_acc_xy, n_batches = 0.0, 0.0, 0.0, 0
    for bx, by, _ in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bx)
        loss, acc_a, acc_c = criterion(out, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item()
        tr_acc_act += acc_a
        tr_acc_xy += acc_c
        n_batches += 1

    scheduler.step()
    model.eval()
    vl_loss, vl_acc_act, vl_acc_xy, v_batches = 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for vx, vy, _ in val_loader:
            vx, vy = vx.to(device), vy.to(device)
            vout = model(vx)
            vloss, vacc_a, vacc_c = criterion(vout, vy)
            vl_loss += vloss.item()
            vl_acc_act += vacc_a
            vl_acc_xy += vacc_c
            v_batches += 1

    tr_loss /= max(1, n_batches)
    vl_loss /= max(1, v_batches)
    print(
        f"Época [{epoch:02d}/{EPOCHS:02d}] | "
        f"Train Loss: {tr_loss:.4f} (Acc Act: {100*tr_acc_act/max(1,n_batches):.1f}%, XY: {100*tr_acc_xy/max(1,n_batches):.1f}%) | "
        f"Val Loss: {vl_loss:.4f} (Acc Act: {100*vl_acc_act/max(1,v_batches):.1f}%, XY: {100*vl_acc_xy/max(1,v_batches):.1f}%)"
    )
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), weights_path)

print(f"\n✅ Entrenamiento finalizado. Mejores pesos guardados en: {weights_path}")

In [ ]:
# 6. Exportar a ONNX Estático (Opset 16, archivo único) y Guardar Dataset Real de Calibración INT8
import onnx
import onnxruntime as ort

model.load_state_dict(torch.load(weights_path, map_location="cpu"))
model.eval().to("cpu")

onnx_path = os.path.join(DRIVE_HAILO_DIR, "sc2_hailo8_transformer.onnx")
calib_path = os.path.join(DRIVE_HAILO_DIR, "sc2_calib_data.npy")

dummy_input = torch.randn(1, IN_CHANNELS, GRID_H, GRID_W, dtype=torch.float32)

export_kwargs = dict(
    export_params=True,
    opset_version=16,
    do_constant_folding=True,
    input_names=["obs_tensor"],
    output_names=["action_chunks_out"],
)
try:
    torch.onnx.export(model, dummy_input, onnx_path, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(model, dummy_input, onnx_path, **export_kwargs)

# Empaquetar pesos dentro del mismo archivo .onnx para evitar archivos .onnx.data externos
onnx_model = onnx.load(onnx_path, load_external_data=True)
onnx.save_model(onnx_model, onnx_path, save_as_external_data=False)
if os.path.exists(onnx_path + ".data"):
    os.remove(onnx_path + ".data")

onnx.checker.check_model(onnx.load(onnx_path))
print(f"✅ Archivo ONNX único exportado y verificado: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")

# Guardar 128 muestras reales balanceadas del set de validación para la cuantización INT8 de Hailo
num_calib = min(128, len(val_ds.states))
calib_samples = val_ds.states[:num_calib].astype(np.float32)
np.save(calib_path, calib_samples)
print(f"✅ Datos reales de calibración guardados: {calib_path} | shape={calib_samples.shape}")

# Prueba rápida con ONNX Runtime
ort_sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
test_out = ort_sess.run(None, {"obs_tensor": calib_samples[:1]})[0]
act_l, tx_l, ty_l, q_l = SC2HailoTransformer.split_predictions(torch.from_numpy(test_out))
print("\n🎯 Inferencia de prueba (Horizonte K=4 Action-Chunks):")
for k in range(ACTION_CHUNK_SIZE):
    a_id = int(act_l[0, k].argmax())
    tx = int(tx_l[0, k].argmax())
    ty = int(ty_l[0, k].argmax())
    q = int(q_l[0, k].argmax())
    print(f"  Paso +{k+1}: [{a_id:02d}] {ID_TO_ACTION[a_id]:<28} -> Celda ({tx:02d}, {ty:02d}) | Queue={bool(q)}")

---
## 🏆 Fase RL (Opcional / Iterativa): Fine-Tuning Actor-Critic PPO ($V_\phi(s_t)$ + Ancla KL) con Partidas Reales
Ejecuta la siguiente celda después de correr `entrenar_rl_hailo8.bat` (`--collect-rl`) en tu PC. Carga las partidas reales guardadas en `/content/drive/MyDrive/hailo_files/sc2_rl_rollouts.npz`, entrena primero una **Red Crítica de Valor $V_\phi(s_t)$** en GPU para estimar la ventaja exacta por estado $\hat{A}_t = G_t - V_\phi(s_t)$ (premiando jugadas acertadas incluso dentro de partidas difíciles), actualiza el **Actor `SC2HailoTransformer`** con penalización $\text{KL}(\pi_\theta \parallel \pi_{\text{ref}})$ estilo **AlphaStar**, y exporta únicamente el Actor a `sc2_hailo8_transformer.onnx` listo para compilar a `.hef`.

> **Tip rápido:** Para hacer solo una actualización de RL sin reentrenar de cero, basta con ejecutar las **Celdas 1, 2, 4**, luego esta **Celda RL (6B)**, y finalmente las celdas de compilación `.hef`.

In [ ]:
# 6B. Fine-Tuning Actor-Critic PPO (Red Crítica V_phi(s_t) + Actor SC2HailoTransformer + Ancla KL)
import copy
import onnx
import onnxruntime as ort
from torch.utils.data import TensorDataset

class SC2ValueCritic(nn.Module):
    """
    Red Crítica de Valor V_phi(s_t) exclusiva para entrenamiento Actor-Critic en Colab.
    Evalúa el estado espacial [B, 12, 16, 16] y predice el retorno esperado G_t.
    No consume recursos en el chip Hailo-8 porque solo el Actor se exporta a .hef.
    """
    def __init__(self, in_channels: int = IN_CHANNELS, hidden_dim: int = 128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, hidden_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim * 16, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
        )

    def forward(self, obs_tensor: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(obs_tensor)).squeeze(-1)


rl_rollout_path = os.path.join(DRIVE_HAILO_DIR, "sc2_rl_rollouts.npz")
weights_path = os.path.join(DRIVE_HAILO_DIR, "sc2_hailo8_transformer_best.pth")
critic_weights_path = os.path.join(DRIVE_HAILO_DIR, "sc2_value_critic.pth")
onnx_path = os.path.join(DRIVE_HAILO_DIR, "sc2_hailo8_transformer.onnx")
calib_path = os.path.join(DRIVE_HAILO_DIR, "sc2_calib_data.npy")
clean_cache_path = os.path.join(DRIVE_HAILO_DIR, "sc2_dynamics_clean_v2_30000.npz")

if not os.path.exists(rl_rollout_path):
    print(f"⚠️ No se encontró {rl_rollout_path}. Ejecuta primero 'entrenar_rl_hailo8.bat' en tu PC.")
else:
    rl_data = np.load(rl_rollout_path)
    rl_states = rl_data["states"].astype(np.float32)
    rl_actions = rl_data["actions"].astype(np.int64)
    rl_returns = rl_data["returns"].astype(np.float32)
    rl_wins = rl_data["wins"] if "wins" in rl_data else np.array([])
    win_rate = (rl_wins.mean() * 100.0) if len(rl_wins) > 0 else 0.0

    print(f"📊 Cargadas {len(rl_states)} muestras RL de {len(rl_wins)} partidas reales (WinRate acumulado: {win_rate:.1f}%)")

    # 1. Entrenar la Red Crítica V_phi(s_t) para estimar el valor base de cada estado tablero 16x16
    critic = SC2ValueCritic(in_channels=IN_CHANNELS, hidden_dim=128).to(device)
    if os.path.exists(critic_weights_path):
        try:
            critic.load_state_dict(torch.load(critic_weights_path, map_location=device))
        except Exception:
            pass

    critic_ds = TensorDataset(torch.from_numpy(rl_states), torch.from_numpy(rl_returns))
    critic_loader = DataLoader(critic_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=(device.type == 'cuda'))
    critic_opt = torch.optim.AdamW(critic.parameters(), lr=1e-3, weight_decay=1e-4)
    CRITIC_EPOCHS = 10

    critic.train()
    for cep in range(1, CRITIC_EPOCHS + 1):
        c_loss_sum, c_steps = 0.0, 0
        for cs, cr in critic_loader:
            cs, cr = cs.to(device), cr.to(device)
            critic_opt.zero_grad()
            v_pred = critic(cs)
            c_loss = F.smooth_l1_loss(v_pred, cr)
            c_loss.backward()
            critic_opt.step()
            c_loss_sum += float(c_loss.item())
            c_steps += 1
    torch.save(critic.state_dict(), critic_weights_path)

    # Calcular ventaja Actor-Critic: A_t = (G_t - V_phi(s_t)) / std
    critic.eval()
    v_preds = []
    with torch.no_grad():
        for i in range(0, len(rl_states), 256):
            batch_s = torch.from_numpy(rl_states[i:i+256]).to(device)
            v_preds.append(critic(batch_s).cpu().numpy())
    v_baseline = np.concatenate(v_preds, axis=0)
    raw_adv = rl_returns - v_baseline
    advantages = (raw_adv - float(raw_adv.mean())) / (float(raw_adv.std()) + 1e-6)
    print(
        f"🧠 Crítico V_phi(s_t) entrenado (Huber Loss: {c_loss_sum/max(1,c_steps):.4f}) | "
        f"Ventaja Actor-Critic A_t: rango [{advantages.min():+.2f}, {advantages.max():+.2f}]"
    )

    # 2. Mezclar un 25% de muestras humanas de alta calidad como ancla supervisada para no olvidar build orders
    if os.path.exists(clean_cache_path):
        sup_cache = np.load(clean_cache_path)
        n_anchor = max(256, len(rl_states) // 3)
        rng_idx = np.random.default_rng(42).choice(len(sup_cache["states"]), size=min(n_anchor, len(sup_cache["states"])), replace=False)
        anchor_states = sup_cache["states"][rng_idx].astype(np.float32)
        anchor_actions = sup_cache["actions"][rng_idx].astype(np.int64)
        anchor_adv = np.ones(len(anchor_states), dtype=np.float32) * 0.85

        all_states = np.concatenate([rl_states, anchor_states], axis=0)
        all_actions = np.concatenate([rl_actions, anchor_actions], axis=0)
        all_adv = np.concatenate([advantages, anchor_adv], axis=0)
        print(f"⚓ Añadidas {len(anchor_states)} secuencias humanas de SC2-Dynamics-50K como ancla de macro.")
    else:
        all_states, all_actions, all_adv = rl_states, rl_actions, advantages

    # 3. Cargar Actor SC2HailoTransformer actual y crear copia congelada de referencia (KL estilo AlphaStar)
    model = SC2HailoTransformer(in_channels=IN_CHANNELS, d_model=128, n_heads=4, num_layers=4).to(device)
    if os.path.exists(weights_path):
        ckpt = torch.load(weights_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt)
        print(f"✅ Pesos del Actor cargados desde: {weights_path}")

    ref_model = copy.deepcopy(model).eval()
    for p in ref_model.parameters():
        p.requires_grad = False

    rl_dataset = TensorDataset(
        torch.from_numpy(all_states),
        torch.from_numpy(all_actions),
        torch.from_numpy(all_adv),
    )
    rl_loader = DataLoader(rl_dataset, batch_size=64, shuffle=True, drop_last=False, num_workers=2, pin_memory=(device.type == 'cuda'))
    rl_optimizer = torch.optim.AdamW(model.parameters(), lr=8e-5, weight_decay=1e-4)
    RL_EPOCHS = 8
    KL_BETA = 0.25

    print(f"🚀 Ejecutando {RL_EPOCHS} épocas de Fine-Tuning Actor-Critic PPO + KL Anchor...")
    for ep in range(1, RL_EPOCHS + 1):
        model.train()
        ep_loss, ep_kl, steps_cnt = 0.0, 0.0, 0
        for bx, by, badv in rl_loader:
            bx, by, badv = bx.to(device), by.to(device), badv.to(device)
            rl_optimizer.zero_grad()

            out = model(bx)
            with torch.no_grad():
                ref_out = ref_model(bx)

            act_l, tx_l, ty_l, q_l = SC2HailoTransformer.split_predictions(out)
            ref_act_l, _, _, _ = SC2HailoTransformer.split_predictions(ref_out)

            # Ponderación exponencial monótona (AWR / MARWIL):
            # A_t > 0 se refuerza (hasta 5.0x); A_t < 0 se suprime suavemente (hasta 0.02x)
            # Nunca genera pesos negativos ni entrena al modelo a imitar decisiones perjudiciales
            sample_weights = torch.clamp(torch.exp(badv / 1.0), min=0.02, max=5.0)

            step_losses = []
            kl_losses = []
            spatial_actions_tensor = torch.tensor(
                [4, 6, 7, 8, 9, 10, 11, 18, 20, 21, 22, 23, 24, 26, 28, 29, 30, 31],
                device=device,
                dtype=torch.int64,
            )
            for k in range(ACTION_CHUNK_SIZE):
                ce_act = F.cross_entropy(act_l[:, k, :], by[:, k, 0], reduction="none")
                ce_tx = F.cross_entropy(tx_l[:, k, :], by[:, k, 1], reduction="none")
                ce_ty = F.cross_entropy(ty_l[:, k, :], by[:, k, 2], reduction="none")
                ce_q = F.cross_entropy(q_l[:, k, :], by[:, k, 3], reduction="none")

                # Máscara espacial: SOLO calcula pérdida en coordenadas (gx, gy) para acciones con target espacial
                # Evita que órdenes económicas como TRAIN_WORKER o HARVEST inyecten ruido (8, 8) al cabezal espacial
                is_spatial = torch.isin(by[:, k, 0], spatial_actions_tensor).float()
                spatial_ce = (ce_tx + ce_ty) * is_spatial

                weighted_ce = (sample_weights * (ce_act + 0.50 * spatial_ce + 0.1 * ce_q)).mean()
                step_losses.append(weighted_ce)

                # Divergencia KL contra la política de referencia para evitar colapso de política
                log_p = F.log_softmax(act_l[:, k, :], dim=-1)
                ref_p = F.softmax(ref_act_l[:, k, :], dim=-1)
                kl = F.kl_div(log_p, ref_p, reduction="batchmean")
                kl_losses.append(kl)

            policy_loss = sum(step_losses) / ACTION_CHUNK_SIZE
            kl_loss = sum(kl_losses) / ACTION_CHUNK_SIZE
            total_loss = policy_loss + KL_BETA * kl_loss

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            rl_optimizer.step()

            ep_loss += float(total_loss.item())
            ep_kl += float(kl_loss.item())
            steps_cnt += 1

        print(f"  [RL Época {ep:02d}/{RL_EPOCHS:02d}] Loss Total: {ep_loss/max(1,steps_cnt):.4f} | KL Divergence: {ep_kl/max(1,steps_cnt):.4f}")

    # 4. Guardar pesos mejorados del Actor y re-exportar ONNX estático + datos de calibración
    torch.save(model.state_dict(), weights_path)
    model.eval().to("cpu")
    dummy_input = torch.randn(1, IN_CHANNELS, GRID_H, GRID_W, dtype=torch.float32)
    export_kwargs = dict(
        export_params=True,
        opset_version=16,
        do_constant_folding=True,
        input_names=["obs_tensor"],
        output_names=["action_chunks_out"],
    )
    try:
        torch.onnx.export(model, dummy_input, onnx_path, dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(model, dummy_input, onnx_path, **export_kwargs)

    onnx_model = onnx.load(onnx_path, load_external_data=True)
    onnx.save_model(onnx_model, onnx_path, save_as_external_data=False)
    if os.path.exists(onnx_path + ".data"):
        os.remove(onnx_path + ".data")

    # Actualizar dataset de calibración INT8 con estados reales de tus partidas RL
    calib_samples = all_states[:min(128, len(all_states))].astype(np.float32)
    np.save(calib_path, calib_samples)
    print(f"\n🎉 ¡Actor RL exportado exitosamente a {onnx_path}! Ahora ejecuta la compilación a .hef abajo.")

---
## 🔧 Compilación del Modelo a `.hef` con Hailo Dataflow Compiler v3.34.0
Las siguientes celdas instalan `condacolab`, crean el entorno Python 3.10 para tu wheel `/content/drive/MyDrive/hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl` y compilan el archivo **`sc2_hailo8_agent.hef`** libre de errores de `.alls`.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Crear entorno Python 3.10 e instalar Hailo Dataflow Compiler 3.34.0 desde tu Google Drive
!apt-get update -qq && apt-get install -y -qq graphviz libgraphviz-dev
!conda create -y -n hailo_env python=3.10
hailo_whl = "/content/drive/MyDrive/hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl"
!conda run -n hailo_env pip install "$hailo_whl"

In [ ]:
%%writefile compile_sc2_hef.py
import os
import numpy as np

# En contenedores de Google Colab la variable USER no existe por defecto y Hailo DFC la necesita en runner.compile()
os.environ.setdefault("USER", "colab")
os.environ.setdefault("LOGNAME", "colab")
os.environ.setdefault("HOME", "/root")

from hailo_sdk_client import ClientRunner

drive_dir = "/content/drive/MyDrive/hailo_files"
onnx_path = os.path.join(drive_dir, "sc2_hailo8_transformer.onnx")
calib_path = os.path.join(drive_dir, "sc2_calib_data.npy")
har_path = os.path.join(drive_dir, "sc2_hailo8_quantized.har")
hef_path = os.path.join(drive_dir, "sc2_hailo8_agent.hef")

print("=== Iniciando Compilación con Hailo Dataflow Compiler (hw_arch=hailo8) ===")
runner = ClientRunner(hw_arch="hailo8")

# Siempre traducimos y cuantizamos el nuevo ONNX entrenado para no reutilizar un .har viejo
if True:
    # 1. Traducir modelo ONNX estático
    runner.translate_onnx_model(
        onnx_path,
        "sc2_hailo8_transformer",
        start_node_names=["obs_tensor"],
        end_node_names=["action_chunks_out"],
        net_input_shapes={"obs_tensor": [1, 12, 16, 16]},
    )
    print("✅ Traducción ONNX completada.")

    # 2. Cargar script de optimización válido para Hailo DFC 3.34.0
    alls_script = "model_optimization_flavor(optimization_level=0, compression_level=0)\n"
    runner.load_model_script(alls_script)
    print("✅ Script de optimización (.alls) aplicado sin errores.")

    # 3. Cuantizar a INT8 usando el dataset de calibración real de StarCraft II
    calib_data = np.load(calib_path).astype(np.float32)
    calib_nhwc = np.transpose(calib_data, (0, 2, 3, 1))
    print(f"🔧 Ejecutando runner.optimize() con {len(calib_nhwc)} muestras reales (shape={calib_nhwc.shape})...")
    try:
        runner.optimize(calib_nhwc)
    except Exception:
        runner.optimize(calib_data)

    runner.save_har(har_path)
    print(f"✅ Modelo cuantizado guardado en: {har_path}")

# 4. Compilar binario físico .hef para el HAT Hailo-8
print("⚙️ Compilando binario .hef para Hailo-8...")
hef = runner.compile()
with open(hef_path, "wb") as f:
    f.write(hef)

print(f"🎉 ¡ÉXITO TOTAL! Binario HEF guardado en Google Drive: {hef_path} ({os.path.getsize(hef_path)/1024:.1f} KB)")

In [ ]:
!USER=colab LOGNAME=colab conda run --no-capture-output -n hailo_env python compile_sc2_hef.py

---
## ⚡ MODO 1-BOTÓN: Fine-Tuning Actor-Critic RL + Exportación ONNX + Compilación `.hef` Automática
A partir de ahora, cada vez que termines de correr `entrenar_rl_hailo8.bat` en tu PC, **solo tienes que darle Play a esta última celda** (incluso si acabas de abrir Colab desde cero):
1. Monta Google Drive automáticamente si aún no está montado.
2. Define las constantes, el **Crítico $V_\phi(s_t)$** y el **Actor `SC2HailoTransformer`**.
3. Entrena el **Actor-Critic PPO + Ancla KL** con las partidas de `sc2_rl_rollouts.npz` y exporta `sc2_hailo8_transformer.onnx`.
4. Prepara automáticamente el entorno Python 3.10 de Hailo DFC v3.34.0 (**sin reiniciar el kernel**) y compila `sc2_hailo8_agent.hef` directo en tu Google Drive.

In [1]:
# 🚀 CELDA TODO-EN-UNO (1 SOLO BOTÓN): Fine-Tuning Actor-Critic RL -> ONNX -> Hailo-8 .HEF
import copy
import hashlib
import os
import shutil
import subprocess
import sys
import time

# 1. Montar Google Drive e instalar onnx/onnxruntime si hace falta
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

try:
    import onnx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"])
    import onnx

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

t_start_all = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DRIVE_HAILO_DIR = "/content/drive/MyDrive/hailo_files"
os.makedirs(DRIVE_HAILO_DIR, exist_ok=True)

# 2. Constantes y Arquitectura SC2HailoTransformer + SC2ValueCritic
GRID_H, GRID_W = 16, 16
IN_CHANNELS = 12
ACTION_CHUNK_SIZE = 4
NUM_ACTIONS = 32
STEP_OUTPUT_DIM = NUM_ACTIONS + GRID_W + GRID_H + 2  # 66

class HailoAttentionBlock2D(nn.Module):
    def __init__(self, d_model: int = 128, n_heads: int = 4, mlp_ratio: int = 2, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5
        self.norm1 = nn.BatchNorm2d(d_model)
        self.q_proj = nn.Conv2d(d_model, d_model, kernel_size=1, bias=True)
        self.k_proj = nn.Conv2d(d_model, d_model, kernel_size=1, bias=True)
        self.v_proj = nn.Conv2d(d_model, d_model, kernel_size=1, bias=True)
        self.out_proj = nn.Conv2d(d_model, d_model, kernel_size=1, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        hidden_dim = d_model * mlp_ratio
        self.norm2 = nn.BatchNorm2d(d_model)
        self.ffn = nn.Sequential(
            nn.Conv2d(d_model, hidden_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv2d(hidden_dim, d_model, kernel_size=1, bias=True),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        N = H * W
        h = self.norm1(x)
        q = self.q_proj(h).reshape(B, self.n_heads, self.head_dim, N).permute(0, 1, 3, 2)
        k = self.k_proj(h).reshape(B, self.n_heads, self.head_dim, N)
        v = self.v_proj(h).reshape(B, self.n_heads, self.head_dim, N).permute(0, 1, 3, 2)
        attn = self.attn_drop(F.softmax(torch.matmul(q, k) * self.scale, dim=-1))
        out = torch.matmul(attn, v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        x = x + self.out_proj(out)
        x = x + self.ffn(self.norm2(x))
        return x

class SC2HailoTransformer(nn.Module):
    def __init__(self, in_channels: int = IN_CHANNELS, d_model: int = 128, n_heads: int = 4, num_layers: int = 4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, d_model, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(d_model),
            nn.ReLU(inplace=True),
        )
        self.pos_embed = nn.Parameter(torch.randn(1, d_model, GRID_H, GRID_W) * 0.02)
        self.blocks = nn.ModuleList([HailoAttentionBlock2D(d_model=d_model, n_heads=n_heads) for _ in range(num_layers)])
        self.final_norm = nn.BatchNorm2d(d_model)
        self.chunk_pool = nn.Sequential(
            nn.Conv2d(d_model, d_model, kernel_size=(GRID_H, ACTION_CHUNK_SIZE), stride=(GRID_H, ACTION_CHUNK_SIZE), bias=False),
            nn.BatchNorm2d(d_model),
            nn.ReLU(inplace=True),
        )
        self.action_head = nn.Conv2d(d_model, STEP_OUTPUT_DIM, kernel_size=1, bias=True)

    def forward(self, obs_tensor: torch.Tensor) -> torch.Tensor:
        x = self.stem(obs_tensor) + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        x = self.final_norm(x)
        return self.action_head(self.chunk_pool(x))

    @staticmethod
    def split_predictions(out_tensor: torch.Tensor):
        seq = out_tensor.squeeze(2).permute(0, 2, 1)
        c1, c2, c3 = NUM_ACTIONS, NUM_ACTIONS + GRID_W, NUM_ACTIONS + GRID_W + GRID_H
        return seq[:, :, :c1], seq[:, :, c1:c2], seq[:, :, c2:c3], seq[:, :, c3:]

class SC2ValueCritic(nn.Module):
    def __init__(self, in_channels: int = IN_CHANNELS, hidden_dim: int = 128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, hidden_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim * 16, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
        )

    def forward(self, obs_tensor: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(obs_tensor)).squeeze(-1)

# 3. Ejecutar Fine-Tuning Actor-Critic PPO + Ancla KL
rl_rollout_path = os.path.join(DRIVE_HAILO_DIR, "sc2_rl_rollouts.npz")
weights_path = os.path.join(DRIVE_HAILO_DIR, "sc2_hailo8_transformer_best.pth")
critic_weights_path = os.path.join(DRIVE_HAILO_DIR, "sc2_value_critic.pth")
onnx_path = os.path.join(DRIVE_HAILO_DIR, "sc2_hailo8_transformer.onnx")
calib_path = os.path.join(DRIVE_HAILO_DIR, "sc2_calib_data.npy")
clean_cache_path = os.path.join(DRIVE_HAILO_DIR, "sc2_dynamics_clean_v2_30000.npz")
hef_path = os.path.join(DRIVE_HAILO_DIR, "sc2_hailo8_agent.hef")

if not os.path.exists(rl_rollout_path):
    raise FileNotFoundError(f"No se encontró {rl_rollout_path}. Ejecuta primero 'entrenar_rl_hailo8.bat' en tu PC.")

rl_data = np.load(rl_rollout_path)
rl_states = rl_data["states"].astype(np.float32)
rl_actions = rl_data["actions"].astype(np.int64)
rl_returns = rl_data["returns"].astype(np.float32)
rl_wins = rl_data["wins"] if "wins" in rl_data else np.array([])
win_rate = (rl_wins.mean() * 100.0) if len(rl_wins) > 0 else 0.0
recent_wr = (rl_wins[-30:].mean() * 100.0) if len(rl_wins) >= 30 else win_rate
print(f"📊 [Paso 1/4] {len(rl_states)} muestras RL de {len(rl_wins)} partidas (WinRate global: {win_rate:.1f}% | Últimas 30: {recent_wr:.1f}%)")

# Entrenar Crítico V_phi(s_t)
critic = SC2ValueCritic(in_channels=IN_CHANNELS, hidden_dim=128).to(device)
if os.path.exists(critic_weights_path):
    try:
        critic.load_state_dict(torch.load(critic_weights_path, map_location=device))
    except Exception:
        pass

critic_ds = TensorDataset(torch.from_numpy(rl_states), torch.from_numpy(rl_returns))
critic_loader = DataLoader(critic_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=(device.type == 'cuda'))
critic_opt = torch.optim.AdamW(critic.parameters(), lr=1e-3, weight_decay=1e-4)
critic.train()
for _ in range(10):
    for cs, cr in critic_loader:
        cs, cr = cs.to(device), cr.to(device)
        critic_opt.zero_grad()
        c_loss = F.smooth_l1_loss(critic(cs), cr)
        c_loss.backward()
        critic_opt.step()
torch.save(critic.state_dict(), critic_weights_path)

critic.eval()
v_preds = []
with torch.no_grad():
    for i in range(0, len(rl_states), 256):
        v_preds.append(critic(torch.from_numpy(rl_states[i:i+256]).to(device)).cpu().numpy())
raw_adv = rl_returns - np.concatenate(v_preds, axis=0)
advantages = (raw_adv - float(raw_adv.mean())) / (float(raw_adv.std()) + 1e-6)
print(f"🧠 [Paso 2/4] Crítico V_phi(s_t) actualizado | Ventaja A_t en [{advantages.min():+.2f}, {advantages.max():+.2f}]")

if os.path.exists(clean_cache_path):
    sup_cache = np.load(clean_cache_path)
    n_anchor = max(256, len(rl_states) // 3)
    rng_idx = np.random.default_rng(42).choice(len(sup_cache["states"]), size=min(n_anchor, len(sup_cache["states"])), replace=False)
    all_states = np.concatenate([rl_states, sup_cache["states"][rng_idx].astype(np.float32)], axis=0)
    all_actions = np.concatenate([rl_actions, sup_cache["actions"][rng_idx].astype(np.int64)], axis=0)
    all_adv = np.concatenate([advantages, np.ones(len(rng_idx), dtype=np.float32) * 0.85], axis=0)
else:
    all_states, all_actions, all_adv = rl_states, rl_actions, advantages

model = SC2HailoTransformer(in_channels=IN_CHANNELS, d_model=128, n_heads=4, num_layers=4).to(device)
if os.path.exists(weights_path):
    ckpt = torch.load(weights_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt)

ref_model = copy.deepcopy(model).eval()
for p in ref_model.parameters():
    p.requires_grad = False

rl_loader = DataLoader(
    TensorDataset(torch.from_numpy(all_states), torch.from_numpy(all_actions), torch.from_numpy(all_adv)),
    batch_size=64,
    shuffle=True,
    drop_last=False,
    num_workers=2,
    pin_memory=(device.type == 'cuda'),
)
rl_optimizer = torch.optim.AdamW(model.parameters(), lr=8e-5, weight_decay=1e-4)
RL_EPOCHS = 8
KL_BETA = 0.25

print(f"🚀 [Paso 3/4] Entrenando Actor SC2HailoTransformer en {device} ({RL_EPOCHS} épocas)...")
for ep in range(1, RL_EPOCHS + 1):
    model.train()
    ep_loss, ep_kl, steps_cnt = 0.0, 0.0, 0
    for bx, by, badv in rl_loader:
        bx, by, badv = bx.to(device), by.to(device), badv.to(device)
        rl_optimizer.zero_grad()
        out = model(bx)
        with torch.no_grad():
            ref_out = ref_model(bx)
        act_l, tx_l, ty_l, q_l = SC2HailoTransformer.split_predictions(out)
        ref_act_l, _, _, _ = SC2HailoTransformer.split_predictions(ref_out)
        sample_weights = torch.clamp(torch.exp(badv / 1.0), min=0.02, max=5.0)
        spatial_actions_tensor = torch.tensor([4, 6, 7, 8, 9, 10, 11, 18, 20, 21, 22, 23, 24, 26, 28, 29, 30, 31], device=device, dtype=torch.int64)
        step_losses, kl_losses = [], []
        for k in range(ACTION_CHUNK_SIZE):
            ce_act = F.cross_entropy(act_l[:, k, :], by[:, k, 0], reduction="none")
            ce_tx = F.cross_entropy(tx_l[:, k, :], by[:, k, 1], reduction="none")
            ce_ty = F.cross_entropy(ty_l[:, k, :], by[:, k, 2], reduction="none")
            ce_q = F.cross_entropy(q_l[:, k, :], by[:, k, 3], reduction="none")
            is_spatial = torch.isin(by[:, k, 0], spatial_actions_tensor).float()
            spatial_ce = (ce_tx + ce_ty) * is_spatial
            step_losses.append((sample_weights * (ce_act + 0.50 * spatial_ce + 0.1 * ce_q)).mean())
            kl_losses.append(F.kl_div(F.log_softmax(act_l[:, k, :], dim=-1), F.softmax(ref_act_l[:, k, :], dim=-1), reduction="batchmean"))
        policy_loss = sum(step_losses) / ACTION_CHUNK_SIZE
        kl_loss = sum(kl_losses) / ACTION_CHUNK_SIZE
        total_loss = policy_loss + KL_BETA * kl_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        rl_optimizer.step()
        ep_loss += float(total_loss.item())
        ep_kl += float(kl_loss.item())
        steps_cnt += 1
    print(f"  [RL Época {ep:02d}/{RL_EPOCHS:02d}] Loss: {ep_loss/max(1,steps_cnt):.4f} | KL: {ep_kl/max(1,steps_cnt):.4f}")

# Exportar ONNX único + datos de calibración INT8
torch.save(model.state_dict(), weights_path)
model.eval().to("cpu")
dummy_input = torch.randn(1, IN_CHANNELS, GRID_H, GRID_W, dtype=torch.float32)
export_kwargs = dict(export_params=True, opset_version=16, do_constant_folding=True, input_names=["obs_tensor"], output_names=["action_chunks_out"])
try:
    torch.onnx.export(model, dummy_input, onnx_path, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(model, dummy_input, onnx_path, **export_kwargs)

onnx_model = onnx.load(onnx_path, load_external_data=True)
onnx.save_model(onnx_model, onnx_path, save_as_external_data=False)
if os.path.exists(onnx_path + ".data"):
    os.remove(onnx_path + ".data")
np.save(calib_path, all_states[:min(128, len(all_states))].astype(np.float32))
print(f"✅ Actor ONNX exportado: {onnx_path}")

# 4. Verificar/Instalar entorno Python 3.10 + Hailo DFC v3.34.0 (SIN reiniciar el kernel) y compilar .HEF
print("🔧 [Paso 4/4] Verificando entorno Hailo Dataflow Compiler v3.34.0 y compilando .hef...")
hailo_py_candidates = [
    "/usr/local/envs/hailo_env/bin/python",
    "/opt/miniconda3/envs/hailo_env/bin/python",
    "/root/miniconda3/envs/hailo_env/bin/python",
]
hailo_py = next((p for p in hailo_py_candidates if os.path.exists(p)), None)

if hailo_py is None:
    print("📦 Configurando Miniconda + hailo_env (Python 3.10) en segundo plano sin reiniciar Colab...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq graphviz libgraphviz-dev", shell=True, check=True)
    conda_bin = shutil.which("conda")
    if conda_bin is None:
        subprocess.run("wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh && bash /tmp/miniconda.sh -b -u -p /opt/miniconda3", shell=True, check=True)
        conda_bin = "/opt/miniconda3/bin/conda"
        subprocess.run(f"{conda_bin} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main --channel https://repo.anaconda.com/pkgs/r || true", shell=True)
    subprocess.run(f"{conda_bin} create -y -n hailo_env python=3.10 -c conda-forge --override-channels", shell=True, check=True)
    hailo_py = next((p for p in hailo_py_candidates if os.path.exists(p)), "/opt/miniconda3/envs/hailo_env/bin/python")
    hailo_whl = "/content/drive/MyDrive/hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl"
    subprocess.run([hailo_py, "-m", "pip", "install", "-q", hailo_whl], check=True)

compile_script = '''import os
import numpy as np
os.environ.setdefault("USER", "colab")
os.environ.setdefault("LOGNAME", "colab")
os.environ.setdefault("HOME", "/root")
from hailo_sdk_client import ClientRunner

drive_dir = "/content/drive/MyDrive/hailo_files"
onnx_path = os.path.join(drive_dir, "sc2_hailo8_transformer.onnx")
calib_path = os.path.join(drive_dir, "sc2_calib_data.npy")
har_path = os.path.join(drive_dir, "sc2_hailo8_quantized.har")
hef_path = os.path.join(drive_dir, "sc2_hailo8_agent.hef")

runner = ClientRunner(hw_arch="hailo8")
runner.translate_onnx_model(
    onnx_path,
    "sc2_hailo8_transformer",
    start_node_names=["obs_tensor"],
    end_node_names=["action_chunks_out"],
    net_input_shapes={"obs_tensor": [1, 12, 16, 16]},
)
runner.load_model_script("model_optimization_flavor(optimization_level=0, compression_level=0)\\n")
calib_data = np.load(calib_path).astype(np.float32)
calib_nhwc = np.transpose(calib_data, (0, 2, 3, 1))
try:
    runner.optimize(calib_nhwc)
except Exception:
    runner.optimize(calib_data)
runner.save_har(har_path)
hef = runner.compile()
with open(hef_path, "wb") as f:
    f.write(hef)
'''
with open("compile_sc2_hef.py", "w", encoding="utf-8") as f:
    f.write(compile_script)

env_vars = os.environ.copy()
env_vars["USER"] = "colab"
env_vars["LOGNAME"] = "colab"
subprocess.run([hailo_py, "compile_sc2_hef.py"], env=env_vars, check=True)

hef_sha = hashlib.sha256(open(hef_path, "rb").read()).hexdigest()[:16].upper()
print(f"\n🎉 ¡PROCESO 1-BOTÓN COMPLETADO EN {time.time() - t_start_all:.1f}s!")
print(f"📦 HEF listo en Drive: {hef_path} ({os.path.getsize(hef_path):,} bytes | SHA256: {hef_sha})")

Mounted at /content/drive
📊 [Paso 1/4] 25000 muestras RL de 316 partidas (WinRate global: 26.6% | Últimas 30: 3.3%)
🧠 [Paso 2/4] Crítico V_phi(s_t) actualizado | Ventaja A_t en [-7.25, +6.33]
🚀 [Paso 3/4] Entrenando Actor SC2HailoTransformer en cuda (8 épocas)...
  [RL Época 01/08] Loss: 3.0590 | KL: 0.3108
  [RL Época 02/08] Loss: 2.8924 | KL: 0.3561
  [RL Época 03/08] Loss: 2.7788 | KL: 0.3805
  [RL Época 04/08] Loss: 2.6920 | KL: 0.4101
  [RL Época 05/08] Loss: 2.5648 | KL: 0.4084
  [RL Época 06/08] Loss: 2.5655 | KL: 0.4299
  [RL Época 07/08] Loss: 2.5530 | KL: 0.4309
  [RL Época 08/08] Loss: 2.4526 | KL: 0.4482


/tmp/ipykernel_581/1572939657.py:255: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy_input, onnx_path, dynamo=False, **export_kwargs)


✅ Actor ONNX exportado: /content/drive/MyDrive/hailo_files/sc2_hailo8_transformer.onnx
🔧 [Paso 4/4] Verificando entorno Hailo Dataflow Compiler v3.34.0 y compilando .hef...
📦 Configurando Miniconda + hailo_env (Python 3.10) en segundo plano sin reiniciar Colab...

🎉 ¡PROCESO 1-BOTÓN COMPLETADO EN 1353.7s!
📦 HEF listo en Drive: /content/drive/MyDrive/hailo_files/sc2_hailo8_agent.hef (3,224,143 bytes | SHA256: F927D851D81677EA)
